In [0]:
from pyspark.sql.types import  StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F 
catalog_name='ecommerce'

In [0]:
df_bronze=spark.table(f'{catalog_name}.bronze.brz_brands')
df_bronze.show(10)

In [0]:
import pyspark.sql.functions as F 

df_silver = df_bronze.withColumn('brand_name', F.trim(F.col('brand_name')))
df_silver.show(10)

In [0]:
df_silver = df_bronze.withColumn('brand_code', F.regexp_replace(F.col('brand_code'), r'[^a-zA-Z0-9]', ''))
df_silver.show(10)

In [0]:
df_silver.select('category_code').distinct().show()

In [0]:
# Anomalies Dictionary
anomalies={
    "GROCERY":"GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}

df_silver = df_silver.replace(anomalies, subset="category_code")
df_silver.select('category_code').distinct().show()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")